In [ ]:
import pickle
# Thay đổi đường dẫn này thành đường dẫn thật tới file pickle của bạn
file_path = 'D:/Data mining/KG-GPT/factkg/data/factkg_test.pickle'
# Mở file với chế độ 'rb' (read binary - đọc dữ liệu nhị phân)
with open(file_path, 'rb') as f:
    data = pickle.load(f)
# In ra dữ liệu đã được load
print(data)

{'I have heard that Mobyland had a successor.': {'Label': [True], 'Entity_set': ['Mobyland'], 'types': ['coll:model', 'existence']}, 'At least Dawn Butler had a successor!': {'Label': [True], 'Entity_set': ['Dawn_Butler'], 'types': ['coll:model', 'existence']}, 'I know that Joseph Brunton had a successor.': {'Label': [True], 'Entity_set': ['Joseph_Brunton'], 'types': ['coll:model', 'existence']}, 'Interestingly, Stubb Cabinet had a successor!': {'Label': [True], 'Entity_set': ['Stubb_Cabinet'], 'types': ['coll:model', 'existence']}, 'Yes and John Sherman Cooper had a successor after him.': {'Label': [True], 'Entity_set': ['John_Sherman_Cooper'], 'types': ['coll:model', 'existence']}, 'Satya Faugoo actually had a successor!': {'Label': [True], 'Entity_set': ['Satya_Faugoo'], 'types': ['coll:model', 'existence']}, 'Well Lamberto V. Avellana had a spouse!': {'Label': [True], 'Entity_set': ['Lamberto_V._Avellana'], 'types': ['coll:model', 'existence']}, 'Catherine Linton had a husband as w

In [8]:
# Lấy ra 5 key đầu tiên
first_5_keys = list(data.keys())[:5]
print("5 keys đầu tiên:", first_5_keys)

print("-" * 50)

# Xem nội dung thực tế của 2 phần tử đầu tiên 
for key in first_5_keys:
    print(f"Key: {key}\nValue: {data[key]}\n")


5 keys đầu tiên: ['I have heard that Mobyland had a successor.', 'At least Dawn Butler had a successor!', 'I know that Joseph Brunton had a successor.', 'Interestingly, Stubb Cabinet had a successor!', 'Yes and John Sherman Cooper had a successor after him.']
--------------------------------------------------
Key: I have heard that Mobyland had a successor.
Value: {'Label': [True], 'Entity_set': ['Mobyland'], 'types': ['coll:model', 'existence']}

Key: At least Dawn Butler had a successor!
Value: {'Label': [True], 'Entity_set': ['Dawn_Butler'], 'types': ['coll:model', 'existence']}

Key: I know that Joseph Brunton had a successor.
Value: {'Label': [True], 'Entity_set': ['Joseph_Brunton'], 'types': ['coll:model', 'existence']}

Key: Interestingly, Stubb Cabinet had a successor!
Value: {'Label': [True], 'Entity_set': ['Stubb_Cabinet'], 'types': ['coll:model', 'existence']}

Key: Yes and John Sherman Cooper had a successor after him.
Value: {'Label': [True], 'Entity_set': ['John_Sherman_C

In [9]:
import pickle
# Thay đổi đường dẫn này thành đường dẫn thật tới file pickle của bạn
file_path = 'D:/Data mining/KG-GPT/factkg/result.pickle'
# Mở file với chế độ 'rb' (read binary - đọc dữ liệu nhị phân)
with open(file_path, 'rb') as f:
    data = pickle.load(f)
# In ra dữ liệu đã được load
print(data)

{1: 'Wrong', 2: 'Wrong', 3: 'Wrong', 4: 'Correct', 5: 'Correct', 6: 'Correct', 7: 'Wrong', 8: 'Correct', 9: 'Wrong', 10: 'Wrong', 11: 'Wrong', 12: 'Wrong', 13: 'Wrong', 14: 'Wrong', 15: 'Correct', 16: 'Correct', 17: 'Wrong', 18: 'Wrong', 19: 'Wrong', 20: 'Correct', 21: 'Wrong', 22: 'Wrong', 23: 'Wrong', 24: 'Wrong', 25: 'Correct', 26: 'Correct', 27: 'Wrong', 28: 'Wrong', 29: 'Wrong', 30: 'Wrong', 31: 'Wrong', 32: 'Correct', 33: 'Wrong', 34: 'Correct', 35: 'Wrong', 36: 'Wrong', 37: 'Correct', 38: 'Correct', 39: 'Correct', 40: 'Wrong', 41: 'Wrong', 42: 'Correct', 43: 'Correct', 44: 'Correct', 45: 'Correct', 46: 'Correct', 47: 'Correct', 48: 'Wrong', 49: 'Wrong', 50: 'Wrong', 51: 'Wrong', 52: 'Wrong', 53: 'Wrong', 54: 'Correct', 55: 'Correct', 56: 'Wrong', 57: 'Wrong', 58: 'Wrong', 59: 'Wrong', 60: 'Wrong', 61: 'Wrong', 62: 'Wrong', 63: 'Correct', 64: 'Wrong', 65: 'Wrong', 66: 'Wrong', 67: 'Wrong', 68: 'Wrong', 69: 'Wrong', 70: 'Wrong', 71: 'Correct', 72: 'Correct', 73: 'Correct', 74: 'Co

In [13]:
"""
FactKG Evaluation Script - Fixed type mapping
Types thực tế: 'existence', 'multi hop', 'negation', 'num1/2/3/4' (= one-hop),
               'multi claim' (= conjunction), 'coll:model', 'coll:presup' (bỏ qua)
"""

import pickle
from collections import defaultdict

TEST_FILE   = "D:/Data mining/KG-GPT/factkg/data/factkg_test.pickle"
RESULT_FILE = "D:/Data mining/KG-GPT/factkg/result.pickle"

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

# ─── MAPPING TYPE ─────────────────────────────────────────────────────────────
# Dựa theo FactKG paper:
#   One-hop     ← num1, num2, num3, num4  (single triple, khác số lượng entity)
#   Conjunction ← multi claim             (nhiều claim kết hợp)
#   Existence   ← existence
#   Multi-hop   ← multi hop
#   Negation    ← negation
#   Bỏ qua      ← coll:model, coll:presup, substitution, written, question
TYPE_NORMALIZE = {
    "num1":        "One-hop",
    "num2":        "One-hop",
    "num3":        "One-hop",
    "num4":        "One-hop",
    "multi claim": "Conjunction",
    "existence":   "Existence",
    "multi hop":   "Multi-hop",
    "negation":    "Negation",
}

DISPLAY_ORDER = ["One-hop", "Conjunction", "Existence", "Multi-hop", "Negation", "Total"]

# ─── LOAD ─────────────────────────────────────────────────────────────────────
print("Loading data...")
test_data   = load_pickle(TEST_FILE)
result_data = load_pickle(RESULT_FILE)
print(f"  Test samples : {len(test_data)}")
print(f"  Result keys  : {len(result_data)}")

# ─── ĐÁNH GIÁ ─────────────────────────────────────────────────────────────────
counts  = defaultdict(lambda: {"correct": 0, "total": 0})
skipped = 0
no_type = 0

for idx, (claim, record) in enumerate(test_data.items(), start=1):
    pred_raw = result_data.get(idx)
    if pred_raw is None:
        skipped += 1
        continue

    correct = (pred_raw.strip().lower() == "correct")

    raw_types = record.get("types", [])
    rtypes = {TYPE_NORMALIZE[t] for t in raw_types if t in TYPE_NORMALIZE}

    if not rtypes:
        no_type += 1
        rtypes = {"Other"}   # coll:model only, question, substitution, written...

    for rtype in rtypes:
        counts[rtype]["correct"] += int(correct)
        counts[rtype]["total"]   += 1

    counts["Total"]["correct"] += int(correct)
    counts["Total"]["total"]   += 1

if skipped:
    print(f"[WARN] Bỏ qua {skipped} sample (thiếu key trong result).")
if no_type:
    print(f"[INFO] {no_type} samples không thuộc 5 type chuẩn → nhóm 'Other'.")

# ─── IN BẢNG KẾT QUẢ ──────────────────────────────────────────────────────────
COL_W = 13
SEP   = "=" * 74

print(f"\n{SEP}")
print("  FactKG Evaluation Results")
print(SEP)
print(f"{'Type':<16}" + "".join(f"{c:>{COL_W}}" for c in DISPLAY_ORDER))
print("-" * 74)

rows_acc, rows_cnt = [], []
for col in DISPLAY_ORDER:
    if col in counts and counts[col]["total"] > 0:
        c   = counts[col]
        acc = 100.0 * c["correct"] / c["total"]
        rows_acc.append(f"{acc:.2f}%")
        rows_cnt.append(f"{c['correct']}/{c['total']}")
    else:
        rows_acc.append("N/A")
        rows_cnt.append("N/A")

print(f"{'Accuracy':<16}" + "".join(f"{v:>{COL_W}}" for v in rows_acc))
print(f"{'Correct/Total':<16}" + "".join(f"{v:>{COL_W}}" for v in rows_cnt))
print(SEP)

# In nhóm Other nếu có
if "Other" in counts:
    c = counts["Other"]
    acc = 100.0 * c["correct"] / c["total"]
    print(f"\n[Other - không thuộc 5 type chuẩn]")
    print(f"  Accuracy      : {acc:.2f}%  ({c['correct']}/{c['total']})")
    print(f"  Bao gồm types : coll:presup, substitution, written, question, ...")

print("\nDone!")

Loading data...
  Test samples : 9041
  Result keys  : 9041

  FactKG Evaluation Results
Type                  One-hop  Conjunction    Existence    Multi-hop     Negation        Total
--------------------------------------------------------------------------
Accuracy               65.06%       69.75%       54.43%       48.14%       74.05%       63.53%
Correct/Total       5037/7742    2297/3293     707/1299     998/2073     973/1314    5744/9041

Done!


In [12]:
import pickle

with open("D:/Data mining/KG-GPT/factkg/data/factkg_test.pickle", "rb") as f:
    data = pickle.load(f)

# Xem tất cả unique types
all_types = set()
for record in data.values():
    for t in record.get("types", []):
        all_types.add(t)

print("Tất cả unique types:", sorted(all_types))

# Xem vài mẫu cụ thể
for i, (claim, record) in enumerate(data.items()):
    print(f"\nClaim: {claim[:60]}")
    print(f"types: {record.get('types')}")
    if i >= 4:
        break

Tất cả unique types: ['coll:model', 'coll:presup', 'existence', 'multi claim', 'multi hop', 'negation', 'num1', 'num2', 'num3', 'num4', 'question', 'substitution', 'written']

Claim: I have heard that Mobyland had a successor.
types: ['coll:model', 'existence']

Claim: At least Dawn Butler had a successor!
types: ['coll:model', 'existence']

Claim: I know that Joseph Brunton had a successor.
types: ['coll:model', 'existence']

Claim: Interestingly, Stubb Cabinet had a successor!
types: ['coll:model', 'existence']

Claim: Yes and John Sherman Cooper had a successor after him.
types: ['coll:model', 'existence']
